# Spatial Voxel Clustering

This notebook compares the legacy voxel clustering workflow based on flat KMeans with a spatially regularized clustering workflow that:

- excludes background voxels that are zero in most compounds,
- clusters voxel profiles in chunks to keep memory bounded,
- injects 3D coordinates so clusters stay volumetric rather than purely feature-driven.

The spatial method returns dense 3D label grids so the downstream Dice and DX export workflow stays compatible.

In [16]:
%load_ext autoreload
%autoreload 2

import pickle
import time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import plotly.express as px

from quartz.data import Grid, pki_grid_scale, set_weights
from quartz.dataprocessing import (
    agglomerative_voxel_clustering,
    calculate_pki_avg,
    dice_coefficient,
    kmeans_clustering,
    spatial_voxel_clustering,
)
from quartz.grids import to_dxgrid

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
PROPS = ["polarizability", "RESP", "vdWpot"]
N_CLUSTERS = 5
MIN_NONZERO_COMPOUNDS = 25
SPATIAL_WEIGHT = 0.45
FEATURE_COMPONENTS = 32
CHUNK_SIZE = 20_000
BATCH_SIZE = 4_096
MIN_COMPONENT_SIZE = 0
BACKGROUND_LABEL = 0

RUN_AGGLOMERATIVE = True
AGGLOMERATIVE_MAX_VOXELS = 12_000
AGGLOMERATIVE_FEATURE_COMPONENTS = 16
AGGLOMERATIVE_LINKAGE = "ward"

NORMALIZED_OPTIONAL_PROPS = ["HOMO_dens", "LUMO_dens", "vdWpot"]
NORMALIZATION_QUANTILE = 0.995
NORMALIZATION_CLIP = True

FRONTIER_ORBS_PATH = Path(
    "/scistor/gt/vie876/proj_FXa_QM/mwfn_frontier/results/frontier_grids.hdf5"
)
VDW_POT_PATH = Path("/scistor/gt/vie876/proj_FXa_QM/mwfn_vdw/results/vdw_grids.hdf5")
EXCEL_PATH = Path("~/proj_FXa_QM/full_database_ligands.xlsx").expanduser()
GRIDS_PATH = Path("grids.pkl")

EXPORT_DX = False
DX_EXPORT_PROP = "polarizability"
DX_EXPORT_PREFIX = "spatial_cluster"
VIS_PROP = "polarizability"
VIS_MAX_POINTS = 50_000

In [18]:
def attach_optional_properties(grids: list[Grid]) -> None:
    if FRONTIER_ORBS_PATH.exists():
        with h5py.File(FRONTIER_ORBS_PATH, "r") as frontier_orbs:
            for grid in grids:
                if grid.id_code not in frontier_orbs:
                    continue
                homo = frontier_orbs[grid.id_code]["HOMO"][()].astype(np.float32)
                homo[homo < 1e-6] = 0.0
                grid.npGrid["HOMO_dens"] = homo

                lumo = frontier_orbs[grid.id_code]["LUMO"][()].astype(np.float32)
                lumo[lumo < 1e-6] = 0.0
                grid.npGrid["LUMO_dens"] = lumo

    if "vdWpot" in PROPS and VDW_POT_PATH.exists():
        with h5py.File(VDW_POT_PATH, "r") as vdw_pot:
            for grid in grids:
                if grid.id_code not in vdw_pot:
                    continue
                vdw = vdw_pot[grid.id_code]["vdWpot"][()].astype(np.float32)
                vdw[np.abs(vdw) < 1e-5] = 0.0
                grid.npGrid["vdWpot"] = vdw


def normalize_loaded_properties(
    grids: list[Grid],
    props: list[str],
    quantile: float = 0.995,
    clip_values: bool = True,
) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []

    for prop in props:
        arrays = [grid.npGrid[prop] for grid in grids if prop in grid.npGrid]
        if not arrays:
            continue

        nonzero_values = [
            arr[np.isfinite(arr) & ~np.isclose(arr, 0.0)]
            for arr in arrays
        ]
        nonzero_values = [vals for vals in nonzero_values if vals.size > 0]
        if not nonzero_values:
            rows.append(
                {
                    "property": prop,
                    "scale": np.nan,
                    "quantile": quantile,
                    "n_nonzero": 0,
                    "max_abs_after": 0.0,
                }
            )
            continue

        stacked = np.concatenate(nonzero_values).astype(np.float32, copy=False)
        scale = float(np.quantile(np.abs(stacked), quantile))
        if not np.isfinite(scale) or scale <= 0.0:
            scale = 1.0

        max_abs_after = 0.0
        for grid in grids:
            if prop not in grid.npGrid:
                continue
            normalized = np.asarray(grid.npGrid[prop], dtype=np.float32) / np.float32(scale)
            if clip_values:
                normalized = np.clip(normalized, -1.0, 1.0)
            normalized[np.isclose(grid.npGrid[prop], 0.0)] = 0.0
            grid.npGrid[prop] = normalized.astype(np.float32, copy=False)
            if normalized.size:
                max_abs_after = max(max_abs_after, float(np.max(np.abs(normalized))))

        rows.append(
            {
                "property": prop,
                "scale": scale,
                "quantile": quantile,
                "n_nonzero": int(stacked.size),
                "max_abs_after": max_abs_after,
            }
        )

    return pd.DataFrame(rows)


def load_doi_map() -> dict[str, str]:
    fx_data = pd.read_excel(EXCEL_PATH).fillna("missing")
    return {
        row["PDB ID"]: row["doi / pdf link"]
        for _, row in fx_data.iterrows()
        if row["doi / pdf link"]
    }


def summarise_cluster_grid(cluster_grid: Grid, method: str) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []
    clustering_meta = cluster_grid.grid_props.get("clustering", {})
    for prop, labels in cluster_grid.npGrid.items():
        label_counts = pd.Series(labels.ravel()).value_counts().sort_index()
        meta = clustering_meta.get(prop, {})
        for label_id, voxel_count in label_counts.items():
            rows.append(
                {
                    "method": method,
                    "property": prop,
                    "label": int(label_id),
                    "voxel_count": int(voxel_count),
                    "fraction": float(voxel_count / labels.size),
                    "active_voxels": int(meta.get("active_voxels", np.count_nonzero(labels))),
                    "explained_variance": float(meta.get("explained_variance", np.nan)),
                }
            )
    return pd.DataFrame(rows)


def dice_vs_pki(cluster_grid: Grid, method: str, grids: list[Grid], pki_avg: Grid) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []
    for prop in PROPS:
        labels = cluster_grid.npGrid[prop]
        for label_id in sorted(np.unique(labels)):
            if method == "spatial" and label_id == BACKGROUND_LABEL:
                continue
            indices = labels == label_id
            if not np.any(indices):
                continue

            masked_grids = [
                Grid(npGrid={prop: g.npGrid[prop][indices]}, grid_props=g.grid_props, mol_data=g.mol_data)
                for g in grids
            ]
            reference = Grid(
                npGrid={prop: pki_avg.npGrid[prop][indices]},
                grid_props=grids[0].grid_props,
            )
            scores = dice_coefficient(reference, masked_grids)[prop]
            df = pd.DataFrame(
                {
                    "code": [g.mol_data.pdb_code for g in grids],
                    "pKi": [g.pki for g in grids],
                    "dice": list(scores.values()),
                }
            ).dropna()
            if len(df) < 2 or df["dice"].nunique() < 2:
                corr = np.nan
            else:
                corr = float(df["dice"].corr(df["pKi"]))
            rows.append(
                {
                    "method": method,
                    "property": prop,
                    "label": int(label_id),
                    "n_voxels": int(np.count_nonzero(indices)),
                    "corr_pki": corr,
                }
            )
    return pd.DataFrame(rows)


def plot_voxel_clusters(cluster_grid: Grid, prop: str, method: str, max_points: int = 50_000):
    if method == "spatial":
        coords = np.argwhere(cluster_grid.npGrid[prop] != BACKGROUND_LABEL)
    else:
        coords = np.argwhere(np.ones_like(cluster_grid.npGrid[prop], dtype=bool))
    label_values = cluster_grid.npGrid[prop][tuple(coords.T)]

    if len(coords) > max_points:
        sampled = np.random.default_rng(42).choice(len(coords), size=max_points, replace=False)
        coords = coords[sampled]
        label_values = label_values[sampled]

    fig = px.scatter_3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        color=label_values.astype(str),
        title=f"{method.capitalize()} clustering for {prop}",
        labels={"x": "X", "y": "Y", "z": "Z", "color": "Cluster"},
        opacity=0.55,
    )
    fig.update_traces(marker={"size": 2})
    fig.show()


def export_cluster_masks(cluster_grid: Grid, prop: str, prefix: str) -> None:
    labels = cluster_grid.npGrid[prop]
    for label_id in sorted(np.unique(labels)):
        if label_id == BACKGROUND_LABEL:
            continue
        mask = np.zeros_like(labels, dtype=np.float32)
        mask[labels == label_id] = 1.0
        to_dxgrid(
            mask,
            origin=cluster_grid.grid_props["origin"],
            spacing=cluster_grid.grid_props["size"],
            filename=f"{prefix}_{prop}_label{label_id}.dx",
        )

In [19]:
doi_map = load_doi_map()

grids: list[Grid] = pickle.load(open(GRIDS_PATH, "rb"))
attach_optional_properties(grids)
normalization_summary = normalize_loaded_properties(
    grids,
    props=NORMALIZED_OPTIONAL_PROPS,
    quantile=NORMALIZATION_QUANTILE,
    clip_values=NORMALIZATION_CLIP,
)

actives = [grid for grid in grids if not np.isnan(grid.pki)]
set_weights(actives, "pki")
for grid in actives:
    if grid.mol_data and grid.mol_data.extra is not None:
        grid.mol_data.extra["doi"] = doi_map.get(grid.id_code, "missing")

pki_scaled_grids = [grid for grid in (pki_grid_scale(grid) for grid in actives) if grid is not None]
print(f"Loaded {len(grids)} grids")
print(f"Using {len(pki_scaled_grids)} pKi-scaled active grids")
print(f"Properties: {PROPS}")
normalization_summary

Loaded 1430 grids
Using 945 pKi-scaled active grids
Properties: ['polarizability', 'RESP', 'vdWpot']


,property,scale,quantile,n_nonzero,max_abs_after
0,HOMO_dens,1.000336e-02,0.995,4981342,1.0
1,LUMO_dens,6.920362e-03,0.995,7506436,1.0
2,vdWpot,3.951941e+06,0.995,307501730,1.0


In [20]:
PROPS.extend(["HOMO_dens", "LUMO_dens", "vdWpot"])

In [21]:
t0 = time.perf_counter()
km_clusters = kmeans_clustering(pki_scaled_grids, props=PROPS, n_clusters=N_CLUSTERS)
km_elapsed = time.perf_counter() - t0

t0 = time.perf_counter()
spatial_clusters = spatial_voxel_clustering(
    pki_scaled_grids,
    props=PROPS,
    n_clusters=N_CLUSTERS,
    min_nonzero_compounds=MIN_NONZERO_COMPOUNDS,
    spatial_weight=SPATIAL_WEIGHT,
    feature_components=FEATURE_COMPONENTS,
    chunk_size=CHUNK_SIZE,
    batch_size=BATCH_SIZE,
    min_component_size=MIN_COMPONENT_SIZE,
    background_label=BACKGROUND_LABEL,
)
spatial_elapsed = time.perf_counter() - t0

agglomerative_clusters = None
agglomerative_elapsed = np.nan
if RUN_AGGLOMERATIVE:
    t0 = time.perf_counter()
    agglomerative_clusters = agglomerative_voxel_clustering(
        pki_scaled_grids,
        props=PROPS,
        n_clusters=N_CLUSTERS,
        min_nonzero_compounds=MIN_NONZERO_COMPOUNDS,
        spatial_weight=SPATIAL_WEIGHT,
        feature_components=AGGLOMERATIVE_FEATURE_COMPONENTS,
        chunk_size=CHUNK_SIZE,
        min_component_size=MIN_COMPONENT_SIZE,
        background_label=BACKGROUND_LABEL,
        max_agglomerative_voxels=AGGLOMERATIVE_MAX_VOXELS,
        linkage=AGGLOMERATIVE_LINKAGE,
    )
    agglomerative_elapsed = time.perf_counter() - t0

summary_frames = [
    summarise_cluster_grid(km_clusters, "kmeans"),
    summarise_cluster_grid(spatial_clusters, "spatial"),
]
if agglomerative_clusters is not None:
    summary_frames.append(summarise_cluster_grid(agglomerative_clusters, "agglomerative"))
cluster_summary = pd.concat(summary_frames, ignore_index=True)

runtime_rows = [
    {"method": "kmeans", "seconds": km_elapsed},
    {"method": "spatial", "seconds": spatial_elapsed},
]
if agglomerative_clusters is not None:
    runtime_rows.append({"method": "agglomerative", "seconds": agglomerative_elapsed})
runtime_summary = pd.DataFrame(runtime_rows)

runtime_summary

,method,seconds
0,kmeans,14.381919
1,spatial,72.537856
2,agglomerative,43.356375


In [6]:
cluster_summary.sort_values(["property", "method", "label"]).head(30)

,method,property,label,voxel_count,fraction,active_voxels,explained_variance
5,kmeans,RESP,0,418390,0.996167,1610,NaN
6,kmeans,RESP,1,277,0.000660,1610,NaN
7,kmeans,RESP,2,879,0.002093,1610,NaN
8,kmeans,RESP,3,192,0.000457,1610,NaN
9,kmeans,RESP,4,262,0.000624,1610,NaN
21,spatial,RESP,0,396254,0.943462,23746,0.483197
22,spatial,RESP,1,1501,0.003574,23746,0.483197
23,spatial,RESP,2,3192,0.007600,23746,0.483197
24,spatial,RESP,3,17186,0.040919,23746,0.483197
25,spatial,RESP,4,749,0.001783,23746,0.483197


In [23]:
pki_avg = calculate_pki_avg(pki_scaled_grids, props=PROPS, n_contribs=50, pre_scaled=False)

corr_frames = [
    dice_vs_pki(km_clusters, "kmeans", pki_scaled_grids, pki_avg),
    dice_vs_pki(spatial_clusters, "spatial", pki_scaled_grids, pki_avg),
]
if agglomerative_clusters is not None:
    corr_frames.append(
        dice_vs_pki(agglomerative_clusters, "agglomerative", pki_scaled_grids, pki_avg)
    )

corr_summary = pd.concat(corr_frames, ignore_index=True)

corr_summary.sort_values(["property", "corr_pki"], ascending=[True, False]).head(30)

,method,property,label,n_voxels,corr_pki
19,kmeans,HOMO_dens,4,350,0.299744
81,agglomerative,HOMO_dens,3,674,0.277671
48,spatial,HOMO_dens,4,590,0.261740
49,spatial,HOMO_dens,5,723,0.253975
16,kmeans,HOMO_dens,1,165,0.216734
83,agglomerative,HOMO_dens,5,174,0.214933
80,agglomerative,HOMO_dens,2,14298,0.164674
18,kmeans,HOMO_dens,3,388,0.150925
15,kmeans,HOMO_dens,0,213799,0.081801
46,spatial,HOMO_dens,2,511,0.060648


In [24]:
plot_voxel_clusters(km_clusters, VIS_PROP, method="kmeans", max_points=VIS_MAX_POINTS)
plot_voxel_clusters(spatial_clusters, VIS_PROP, method="spatial", max_points=VIS_MAX_POINTS)
if agglomerative_clusters is not None:
    plot_voxel_clusters(
        agglomerative_clusters,
        VIS_PROP,
        method="agglomerative",
        max_points=VIS_MAX_POINTS,
    )

if EXPORT_DX:
    export_cluster_masks(spatial_clusters, DX_EXPORT_PROP, DX_EXPORT_PREFIX)